# Formal Verification with Axiom + `@contract`

Each contract's post-condition can now delegate LLM outputs to Axiom's Lean4 engine for formal verification before accepting them.

**Flow:**
1. The `@contract` decorator calls the LLM to generate a Lean4 proof
2. `post()` sends the proof to Axiom (Axle SDK) for verification against Lean4's type checker
3. If verification fails, the contract retries with error feedback — up to N attempts

**Prerequisites:**
```bash
pip install symbolicai[lean]
```
Set `FORMAL_ENGINE_API_KEY` and `FORMAL_ENGINE=axiom` in your `symai.config.json`.

In [1]:
import os

from pydantic import Field

from symai import Expression, Interface
from symai.models.base import LLMDataModel
from symai.strategy import contract

## 1. Data Models

The input carries the formal statement (with `sorry` as a placeholder) and an optional hint for the LLM. The output is the complete Lean4 proof.

In [2]:
class InpProveTheorem(LLMDataModel):
    formal_statement: str = Field(
        description=(
            "A complete Lean4 theorem signature with `sorry` as the proof body. "
            "Example: 'theorem my_thm (n : Nat) : 0 + n = n := by sorry'. "
            "The LLM must preserve this exact name and signature."
        ),
    )
    hint: str = Field(
        description=(
            "Optional tactic hint for the LLM, e.g. 'Use omega' or 'Try induction on n'. "
            "Left empty if the LLM should figure out the proof strategy on its own."
        ),
    )


class OutpProveTheorem(LLMDataModel):
    proof: str = Field(
        description=(
            "The complete Lean4 theorem with a valid tactic proof replacing `sorry`. "
            "Must use the EXACT same theorem name and type signature from the formal_statement. "
            "Must NOT include any import statements — the environment already has Mathlib."
        ),
    )

## 2. Contract

The `@contract` decorator orchestrates the loop: it calls the LLM (using the `prompt` property and the data models above), then runs `post()` to validate. If `post()` raises, the contract feeds the error back to the LLM and retries.

In [3]:
CONTRACT_KWARGS = {
    "pre_remedy": False,
    "post_remedy": True,
    "verbose": True,
    "remedy_retry_params": {
        "tries": 8,
        "delay": 0.5,
        "max_delay": 0.0,
        "backoff": 0.5,
        "jitter": 0.0,
        "graceful": False,
    },
}


@contract(**CONTRACT_KWARGS)
class ProveTheorem(Expression):
    """LLM writes a Lean4 proof; Axiom verifies it in post()."""

    def __init__(self, formal_statement: str, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._formal_statement = formal_statement

    @property
    def prompt(self) -> str:
        return (
            "You are a Lean4 proof assistant.\n\n"
            "Given a formal statement (with `sorry` as placeholder), replace `sorry` with a real proof.\n"
            "You MUST use the EXACT same theorem name and signature from the formal_statement.\n"
            "Return ONLY the complete theorem with the proof filled in, no markdown fences.\n"
            "Do NOT import anything — the environment already has Mathlib available.\n"
        )

    def post(self, output: OutpProveTheorem) -> bool:
        axiom = Interface("axiom")
        result = axiom(
            output.proof,
            tool="verify_proof",
            config={"formal_statement": self._formal_statement},
        )
        if not result.raw["okay"]:
            errors = result.raw.get("lean_messages", {}).get("errors", [])
            msg = f"Proof verification failed. Lean errors: {errors}"
            raise ValueError(msg)
        return True

    def forward(self, _input: InpProveTheorem, **_kwargs) -> OutpProveTheorem:
        if not self.contract_successful:
            raise self.contract_exception or ValueError(
                f"Contract validation failed after {CONTRACT_KWARGS['remedy_retry_params']['tries']} tries.",
            )
        return self.contract_result

## 3. Run it

We ask the LLM to prove that natural number addition is commutative. The `formal_statement` carries the theorem signature with `sorry` — the LLM fills in the real proof, and Axiom verifies it against Lean4.

In [4]:
formal_statement = "theorem my_add_comm (a b : Nat) : a + b = b + a := by sorry"

prover = ProveTheorem(formal_statement=formal_statement)
result = prover(InpProveTheorem(
    formal_statement=formal_statement,
    hint="Use the omega tactic or Nat.add_comm",
))

print("Verified proof:")
print(result.proof)

2026-03-06 23:57:52.051 | INFO     | symai.strategy:__init__:792 - Initializing contract...
2026-03-06 23:57:52.052 | INFO     | symai.strategy:__init__:805 - Contract initialization complete!
2026-03-06 23:57:52.054 | INFO     | symai.strategy:_start_contract_execution:810 - Starting contract execution...
2026-03-06 23:57:52.055 | INFO     | symai.strategy:_validate_input:613 - Starting input validation...
2026-03-06 23:57:52.056 | INFO     | symai.strategy:_validate_input:656 - Skip; no pre-condition validation was required!
2026-03-06 23:57:52.057 | INFO     | symai.strategy:_validate_output:660 - Starting output validation...
2026-03-06 23:57:52.057 | INFO     | symai.strategy:_validate_output:670 - Getting a valid output type...
2026-03-06 23:57:52.058 | INFO     | symai.strategy:forward:458 - Initializing validation…


╭─────────────────────────────────────────── Prompt ───────────────────────────────────────────╮
│                                                                                              │
│  You are a Lean4 proof assistant.                                                            │
│                                                                                              │
│  Given a formal statement (with `sorry` as placeholder), replace `sorry` with a real proof.  │
│  You MUST use the EXACT same theorem name and signature from the formal_statement.           │
│  Return ONLY the complete theorem with the proof filled in, no markdown fences.              │
│  Do NOT import anything — the environment already has Mathlib available.                     │
│                                                                                              │
│                                                                                              │
╰──────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Input data model ────────────────────────────────────────────────╮
│                                                                                                                 │
│  [[Schema]]                                                                                                     │
│  - "formal_statement" (string, required)                                                                        │
│  - "hint" (string, required)                                                                                    │
│                                                                                                                 │
│  [[Definitions]]                                                                                                │
│  - InpProveTheorem:                                                                                             │
│    - "formal_statement": A complete Lean4 theorem signature with `sorry` as the proof body. Example: 'theorem   │
│  my_thm (n : Nat) : 0 + n = n := by sorry'. The LLM must preserve this exact name and signature.                │
│    - "hint": Optional tactic hint for the LLM, e.g. 'Use omega' or 'Try induction on n'. Left empty if the LLM  │
│  should figure out the proof strategy on its own.                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── Output data model ───────────────────────────────────────────────╮
│                                                                                                                 │
│  [[Schema]]                                                                                                     │
│  - "proof" (string, required)                                                                                   │
│                                                                                                                 │
│  [[Definitions]]                                                                                                │
│  - OutpProveTheorem:                                                                                            │
│    - "proof": The complete Lean4 theorem with a valid tactic proof replacing `sorry`. Must use the EXACT same   │
│  theorem name and type signature from the formal_statement. Must NOT include any import statements — the        │
│  environment already has Mathlib.                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

2026-03-06 23:57:54.084 | INFO     | symai.strategy:forward:474 - Prepared 9 remedy seeds for validation attempts…
2026-03-06 23:57:54.085 | INFO     | symai.strategy:_run_validation_attempts:388 - Attempt 1/8: Attempting validation…
2026-03-06 23:57:54.086 | SUCCESS  | symai.strategy:forward:488 - Validation completed successfully!
2026-03-06 23:57:54.087 | SUCCESS  | symai.strategy:_validate_output:681 - Type successfully created!
2026-03-06 23:57:54.088 | INFO     | symai.strategy:_validate_output:684 - Validating post-conditions with remedy...
2026-03-06 23:57:54.725 | SUCCESS  | symai.strategy:_validate_output:694 - Post-condition validation successful!
2026-03-06 23:57:54.726 | INFO     | symai.strategy:_execute_forward_call:926 - Executing original forward method...
2026-03-06 23:57:54.727 | SUCCESS  | symai.strategy:_finalize_contract_output:967 - Contract validation successful!


Verified proof:
theorem my_add_comm (a b : Nat) : a + b = b + a := by omega


## 4. Using Axiom directly

You can also use the Axiom interface standalone — without a contract — for quick checks.

In [5]:
axiom = Interface("axiom")

# Check valid Lean4 code
valid = axiom(
    "theorem hello (a b : Prop) (ha : a) (hb : b) : a ∧ b := by exact ⟨ha, hb⟩",
    tool="check",
)
print("Valid check:", valid.raw["okay"])  # True

# Check invalid Lean4 code
invalid = axiom(
    "theorem broken (a : Prop) : a := by sorry_not_a_tactic",
    tool="check",
)
print("Invalid check:", invalid.raw["okay"])  # False

# Extract theorems from a file
extracted = axiom(
    "theorem my_thm (n : Nat) : 0 + n = n := by omega",
    tool="extract_theorems",
)
print("Extracted documents:", list(extracted.raw["documents"].keys()))

Valid check: True
Invalid check: False
Extracted documents: ['my_thm']
